## Fairness metrics Overview

### Composite Unfairness Score

$$\text{Unfairness} = \alpha \cdot DP + \beta \cdot EO + \gamma \cdot CAL$$

where $\alpha + \beta + \gamma = 1$ and all weights are non-negative. The score is an **unfairness score** — higher means less fair. Since DP, EO, and CAL are each individually bounded in $[0,1]$ (see below), and $\alpha+\beta+\gamma=1$, the composite is a convex combination of bounded quantities and is itself bounded in $[0,1]$.

---

### Demographic Parity (DP)

$$DP = \frac{1}{A}\sum_{a=1}^A \left[\frac{1}{G_a - 1}\sum_{g=2}^{G_a} \left| P(\hat{y}=1 \mid g,a) - P(\hat{y}=1 \mid 1,a) \right| \right]$$

Measures whether approval rates differ across subgroups. $g=1$ denotes the reference subgroup for attribute $a$.

The inner average over $g$ divides by $G_a - 1$ (the number of non-reference subgroups) rather than summing raw gaps. Without this, an attribute with more subgroups (e.g. race, with 5 non-reference groups) would mechanically inflate its contribution relative to an attribute with fewer subgroups (e.g. sex, with 2 non-reference groups) — not because the underlying disparities are larger, but purely because there are more terms in the sum. Dividing by $G_a-1$ converts each attribute's contribution into an *average* gap, removing this artifact of subgroup count.

The outer average over $a$ divides by $A$ for the same reason one level up, and additionally ensures $DP \in [0,1]$: each inner bracket is bounded in $[0,1]$ (an average of probability differences, each individually in $[0,1]$), so summing $A$ such brackets gives $[0,A]$, and dividing by $A$ rescales to $[0,1]$.

This construction assumes $A$ — the set of protected attributes included — is identical across DP, EO, and CAL. We hold this fixed deliberately: selectively excluding a protected attribute from one metric is a structurally detectable form of gaming (a regulator can simply ask "why is sex absent from your EO calculation?"), addressable by a basic completeness requirement. This is qualitatively different from the weighting manipulation ($\alpha$, $\beta$, $\gamma$, and $B$ — see below) that this dissertation is concerned with, where every choice of parameter still produces a number that looks like a legitimate fairness score.

---

### Equalised Odds (EO)

$$EO = \frac{1}{A}\sum_{a=1}^A \left[\frac{1}{G_a - 1}\sum_{g=2}^{G_a} \frac{1}{2}\left( \left| P(\hat{y}=1 \mid y=1,g,a) - P(\hat{y}=1 \mid y=1,1,a) \right| + \left| P(\hat{y}=1 \mid y=0,g,a) - P(\hat{y}=1 \mid y=0,1,a) \right| \right) \right]$$

Measures whether TPR and FPR differ across subgroups. The two terms inside the parentheses capture the TPR gap and FPR gap respectively.

The same $\frac{1}{G_a-1}$ and $\frac{1}{A}$ normalisations apply here as in DP, and for the same reasons. EO additionally requires a $\frac{1}{2}$ that DP does not: each inner term sums *two* gaps (TPR and FPR), each individually bounded in $[0,1]$, so their sum is bounded in $[0,2]$ rather than $[0,1]$. The $\frac12$ averages the TPR gap and FPR gap together, bringing the inner bracket back to $[0,1]$ and keeping it on the same scale as DP's inner bracket — necessary for $DP$, $EO$, and $CAL$ to be comparable once combined via $\alpha$, $\beta$, $\gamma$. Placing the $\frac12$ at this innermost level (rather than factoring it out front) keeps every layer of the expression independently interpretable: the bracket is "the average of the TPR gap and FPR gap for subgroup $g$," not an arbitrary partial sum awaiting a distant normalising constant.

---

### Calibration (CAL)

$$CAL = \frac{1}{A}\sum_{a=1}^A \left[\frac{1}{G_a - 1}\sum_{g=2}^{G_a} \left(\frac{1}{B}\sum_{b=1}^B \left| P(\hat{y}=1 \mid b,g,a) - P(\hat{y}=1 \mid b,1,a) \right| \right) \right]$$

Measures whether predicted probabilities reflect actual outcomes equally across subgroups.

The innermost average over $b$ divides by $B$, the number of calibration bins, applying the same bounding logic as the other layers: each bin-level gap is a probability difference bounded in $[0,1]$, so averaging over $B$ bins keeps the result in $[0,1]$. The same $\frac{1}{G_a-1}$ and $\frac{1}{A}$ normalisations then apply exactly as in DP and EO. Unlike EO, no additional rescaling constant is needed beyond the three averaging layers, since CAL's $B$ terms are repetitions of the same quantity (a gap, evaluated per bin) rather than two structurally different quantities (TPR gap and FPR gap) that need reconciling into one comparable unit.

**$B$ is itself a free, tuneable parameter**, set by whoever implements the monitoring system, with no value specified or implied by regulation. Unlike $\alpha,\beta,\gamma$, which form a continuous 2-dimensional simplex, $B$ is a positive integer with no natural upper bound, so sensitivity to $B$ is not the same kind of exercise as sensitivity to $(\alpha,\beta,\gamma)$ and should be examined separately. Too few bins make CAL uninformative (coarse probability ranges hide real disparities); too many make bins sparse, especially for smaller subgroups, producing noisy and unstable estimates. Because CAL is itself sensitive to $B$, $\gamma$'s *effective* influence on the composite score is partly a function of which $B$ was chosen — meaning a robustness check that sweeps $(\alpha,\beta,\gamma)$ and a robustness check that sweeps $B$ are not fully independent and may need to be considered jointly.

---

### Reference Groups

| Attribute | Reference group |
|---|---|
| Race | White |
| Ethnicity | Not Hispanic or Latino |
| Sex | Male |

---

### Notes

- $a$ denotes a protected attribute; $A$ is the total number of protected attributes
- $g$ indexes subgroups within attribute $a$; $G_a$ is the number of subgroups for attribute $a$
- $g=1$ is the reference subgroup for each attribute (see table above)
- $B$ is the number of calibration bins (e.g. 10 equal-width bins of 0–10%, 10–20%, ..., 90–100%). $B$ is a free parameter not specified by regulation; results should be checked for sensitivity to this choice (see CAL discussion above)
- $DP$, $EO$, and $CAL$ are each individually normalised to $[0,1]$, via averaging (not summing) across bins, subgroups, and attributes at every layer. This means each metric is independently interpretable as a percentage on its own, in addition to being combinable into the composite score
- The three-layer normalisation ($\frac1B$ where applicable, $\frac{1}{G_a-1}$, $\frac1A$) is a scale-correction, not a substantive weighting choice: it removes artifacts of subgroup count and bin count so that each metric reflects average disparity rather than being inflated by how many categories happen to exist. This is conceptually distinct from $\alpha$, $\beta$, $\gamma$, which **are** a deliberate, substantive importance judgement across DP, EO, and CAL
- The three metrics are in tension by construction: when base rates differ across groups, DP, EO, and CAL cannot all equal zero simultaneously (Chouldechova, 2017). The weights $\alpha$, $\beta$, $\gamma$ therefore encode a substantive choice about which form of unfairness is prioritised — and, as with $B$, this choice is left entirely open by the regulatory text

### A. Demographic parity

Does the model approve applicants at equal rates across groups?

$$DP = \frac{1}{A}\sum_{a=1}^A \left[\frac{1}{G_a - 1}\sum_{g=2}^{G_a} \left| P(\hat{y}=1 \mid g,a) - P(\hat{y}=1 \mid 1,a) \right| \right]$$

In [ ]:
dict_references = {"race": 4, "sex": 0, "ethnicity": 0} # Reference groups (white, male, non-Hispanic or Latino)

def dem_parity_calc(attribute):

    dem_parity = (
        hmda_2007_pred
        .groupby(attribute)['predictions']
        .value_counts(normalize=True)
        .unstack(fill_value = 0)
        .round(2)
    )

    dem_parity_values = []

    for i in range(len(dem_parity)):
        if i != dict_references[attribute]:
            dem_parity_values.append(abs(dem_parity.iloc[i][1] - dem_parity.iloc[dict_references[attribute]][1]))

    return(sum(dem_parity_values) / (len(dem_parity)-1))

In [ ]:
def dem_parity_total():
    race_dem_parity = dem_parity_calc(attribute = "race")
    sex_dem_parity = dem_parity_calc(attribute = "sex")
    ethnicity_dem_parity = dem_parity_calc(attribute = "ethnicity")

    return((race_dem_parity + sex_dem_parity + ethnicity_dem_parity) / len(dict_references))

In [ ]:
print(round(dem_parity_total(),2))